# CIGALE Decomposition Validation: Fixing the Quiescent-Region Asymmetry

`CIGALE_Decomposition_Validation.ipynb` found that reconstructing real
galaxies' full SEDs as host (CIGALE-decomposed) + the paper's fixed Type 1
(face-on) or Type 2 (edge-on) SKIRTOR template gives high *overall* UVJ
classification agreement (~95%) but hides a sharp, opposite-direction
failure in the Quiescent region specifically - the region the paper's
"hidden quiescent galaxies" result depends on:

| True region | Type 1 agreement | Type 2 agreement |
|---|---|---|
| Quiescent | 72.6% (misses real quiescent galaxies) | 100% (but 4.5% false-positive rate elsewhere) |
| Star-forming | 98.8% | 95.7% |
| Dusty | 84.8% | 77.3% |

**Root cause (diagnosed below, not assumed):** the paper's Type 1/Type 2
templates use SKIRTOR torus shape parameters `p=0.5, q=0` at the two most
extreme possible inclinations (`i=0`/face-on and `i=90`/edge-on). CIGALE's
own SED-fitting run that produced the real decompositions used a
*completely different* torus shape (`p=1.0, q=1.0`) and never explored
either extreme inclination at all - scanning all 6509 galaxies' FITS
headers below shows CIGALE only ever fit `i=30` or `i=70`. So the paper's
theoretical templates are, in a precise and quantifiable sense, the wrong
AGN model relative to what was actually fit to these galaxies - not just an
approximation of it.

**The fix tested here:** reconstruct each galaxy using CIGALE's *own*
best-fit SKIRTOR geometry (read straight from that galaxy's FITS header)
instead of the paper's fixed Type 1/Type 2 grid. Because CIGALE's fitting
run only ever used two discrete geometries, this needs only two SKIRTOR
template reads total (same cost as Type1/Type2) - just assigned to the
*correct* galaxies instead of applied uniformly to all of them.

This notebook is additive: it does not modify
`CIGALE_Decomposition_Validation.ipynb`. All three reconstruction methods
(Type1, Type2, Matched) are computed here for a direct, apples-to-apples
comparison.

In [ ]:
import sys
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath('..'))

from src import config
from glass import data_io, composite_math, photometry, visualization, analysis
from astropy.io import fits as _fits

plt.style.use('default')
visualization.apply_pasa_style()
os.makedirs(config.PROCESSED_DATA_DIR, exist_ok=True)

FIX_OUTPUT_DIR = os.path.join(config.PROCESSED_DATA_DIR, 'cigale_geometry_fix')
os.makedirs(FIX_OUTPUT_DIR, exist_ok=True)

METHOD_COLORS = {'Type1': '#1A6FB5', 'Type2': '#CC2929', 'Matched': '#2E8B57'}

## 1. Population and the SKIRTOR-geometry diagnosis

Loads the same `fracAGN > 0` population as the original validation notebook,
then scans every galaxy's CIGALE FITS header for its best-fit SKIRTOR
geometry (`agn.t`, `agn.pl`, `agn.q`, `agn.oa`, `agn.R`, `agn.i`) - cached to
disk since it's a full population scan (header-only reads, so this is much
cheaper than reading the full spectral tables).

In [ ]:
cigale_csv = os.path.join(config.RAW_DATA_DIR, 'full_zfourge_decomposed', 'zfourge_full_final.csv')
agn_frac_csv = os.path.join(config.RAW_DATA_DIR, 'full_zfourge_decomposed', 'agn_fractions.csv')

df_cig = pd.read_csv(cigale_csv, low_memory=False)
z_col = 'zpk_x' if 'zpk_x' in df_cig.columns else 'zpk'
df_cig = df_cig.merge(pd.read_csv(agn_frac_csv), on='ID', how='left')

AGN_FRAC_MIN = 0.0
df_agn = df_cig[df_cig['fracAGN'] > AGN_FRAC_MIN].reset_index(drop=True)
print(f"AGN-hosting galaxies (fracAGN > {AGN_FRAC_MIN}): {len(df_agn)} of {len(df_cig)}")


def _fits_path(gid, field):
    fits_num = gid.split('_', 1)[1]
    return os.path.join(config.RAW_DATA_DIR, 'full_zfourge_decomposed',
                         f"{field.lower()}_best_models_fits", f"{fits_num}_best_model.fits")


GEOMETRY_CSV = os.path.join(FIX_OUTPUT_DIR, 'agn_geometry.csv')
if os.path.exists(GEOMETRY_CSV):
    geom_df = pd.read_csv(GEOMETRY_CSV)
else:
    rows = []
    t0 = time.time()
    for _, r in df_agn.iterrows():
        path = _fits_path(r['ID'], r['field'])
        if not os.path.exists(path):
            continue
        h = _fits.getheader(path, 1)
        rows.append({'ID': r['ID'], 't': h['agn.t'], 'p': h['agn.pl'], 'q': h['agn.q'],
                     'oa': h['agn.oa'], 'R': h['agn.R'], 'i': h['agn.i']})
    geom_df = pd.DataFrame(rows)
    geom_df.to_csv(GEOMETRY_CSV, index=False)
    print(f"Scanned {len(geom_df)} headers in {time.time() - t0:.0f}s")

df_agn = df_agn.merge(geom_df, on='ID', how='left')

print("\nSKIRTOR geometry combinations CIGALE actually fit, across all "
      f"{len(df_agn)} AGN-host galaxies:")
print(df_agn.groupby(['t', 'p', 'q', 'oa', 'R', 'i']).size().sort_values(ascending=False))
print(f"\nPaper's Type1/Type2 templates use: p={config.SKIRTOR_TYPE1_PARAMS['p']}, "
      f"q={config.SKIRTOR_TYPE1_PARAMS['q']}, i={config.SKIRTOR_TYPE1_PARAMS['inclination']} "
      f"(Type1) / i={config.SKIRTOR_TYPE2_PARAMS['inclination']} (Type2)")

The table above confirms the diagnosis directly from the data: every
one of the 6509 AGN-host galaxies was fit with the *same* torus shape
(`t=7, p=1.0, q=1.0, oa=40, R=20`) and *only two* inclinations (`i=30` or
`i=70`) - never `p=0.5, q=0` and never the face-on/edge-on extremes the
paper's Type1/Type2 templates use. `agn.i` alone therefore determines the
correct AGN template for every galaxy.

In [ ]:
skirtor_dir = os.path.join(config.RAW_DATA_DIR, 'Templates', 'Skirtor')
agn_type1 = data_io.read_skirtor_model(skirtor_dir, **config.SKIRTOR_TYPE1_PARAMS)
agn_type2 = data_io.read_skirtor_model(skirtor_dir, **config.SKIRTOR_TYPE2_PARAMS)

MATCHED_GEOMETRY_PARAMS = {'optical_depth': 7, 'p': 1, 'q': 1, 'opening_angle': 40, 'radius_ratio': 20}
skirtor_matched_i30 = data_io.read_skirtor_model(skirtor_dir, inclination=30, **MATCHED_GEOMETRY_PARAMS)
skirtor_matched_i70 = data_io.read_skirtor_model(skirtor_dir, inclination=70, **MATCHED_GEOMETRY_PARAMS)
MATCHED_TEMPLATES = {30: skirtor_matched_i30, 70: skirtor_matched_i70}

FIXED_TEMPLATES = {'Type1': agn_type1, 'Type2': agn_type2}
filters = photometry.load_passbands(config.FILTER_PATHS)

print("Templates loaded: Type1 (paper, face-on), Type2 (paper, edge-on), "
      "Matched i=30 (CIGALE's own fit, 76% of galaxies), Matched i=70 (CIGALE's own fit, 24% of galaxies)")

**Column-reuse note (as in the original validation notebook):**
`data_io.read_cigale_best_model()`'s Fnu-derived `'Total Flux
(erg/s/cm^2/Angstrom)'` column is deliberately overwritten by
`analysis.decompose_cigale_sed(..., target='host')`'s `L_lambda_total`-based
value - the ground-truth "full" SED for every residual below is read from
`L_lambda_total` directly, before decomposition. Relative flux residuals
are only evaluated where true flux exceeds `1e-4` of that galaxy's peak
`L_lambda_total` and restframe wavelength exceeds `1300` A (excludes the
Lyman-continuum/IGM-absorption regime that no naive additive model - fixed
or matched - replicates, and which plays no role in the U/V/J passbands).

In [ ]:
FLUX_FLOOR_FRACTION = 1e-4
WAVELENGTH_VALID_MIN = 1300.0


def process_galaxy(gid, field, z, fracAGN, agn_i):
    """
    Reconstructs galaxy `gid`'s full SED via three methods - Type1, Type2
    (paper's fixed templates) and Matched (CIGALE's own best-fit geometry
    for this galaxy, selected by agn_i) - and returns per-method summary
    statistics for direct comparison.
    """
    path = _fits_path(gid, field)
    if not os.path.exists(path):
        return None

    full_sed = data_io.read_cigale_best_model(path, redshift=z, restframe=True)
    wl_full = full_sed['lambda (Angstroms)'].values.astype(float)
    full_L = full_sed['L_lambda_total'].values.astype(float)
    peak_L = np.nanmax(full_L)
    floor = FLUX_FLOOR_FRACTION * peak_L if peak_L > 0 else 0.0

    host_sed = analysis.decompose_cigale_sed(full_sed, target='host')
    alpha_theory = fracAGN / (1.0 - fracAGN) if fracAGN < 1.0 else np.nan

    methods = dict(FIXED_TEMPLATES)
    matched_key = int(round(agn_i)) if np.isfinite(agn_i) else None
    if matched_key in MATCHED_TEMPLATES:
        methods['Matched'] = MATCHED_TEMPLATES[matched_key]

    row = {'ID': gid, 'field': field, 'fracAGN': fracAGN, 'redshift': z, 'agn_i': agn_i,
           'alpha_theory': alpha_theory}

    for mname, agn_template in methods.items():
        composite = composite_math.create_composite_sed(agn_template, host_sed, alpha_theory)
        wl_c = composite['lambda (Angstroms)'].values
        flux_c = composite['Total Flux (erg/s/cm^2/Angstrom)'].values

        full_interp = np.interp(wl_c, wl_full, full_L, left=np.nan, right=np.nan)
        valid = np.isfinite(full_interp) & (full_interp > floor) & (wl_c > WAVELENGTH_VALID_MIN)
        resid = (flux_c[valid] - full_interp[valid]) / full_interp[valid] if valid.any() else np.array([])
        row[f'resid_median_{mname}'] = np.median(resid) if len(resid) else np.nan
        row[f'max_abs_resid_{mname}'] = np.max(np.abs(resid)) if len(resid) else np.nan

        try:
            uv_r, vj_r = photometry.calculate_UVJ_colours(composite, filters['U'], filters['V'], filters['J'])
            cls_r = int(photometry.classify_uvj(np.array([vj_r]), np.array([uv_r]))[0])
        except (ValueError, ZeroDivisionError):
            uv_r, vj_r, cls_r = np.nan, np.nan, -1
        row[f'UV_recombined_{mname}'] = uv_r
        row[f'VJ_recombined_{mname}'] = vj_r
        row[f'cls_recombined_{mname}'] = cls_r

    return row

In [ ]:
SUMMARY_CSV = os.path.join(FIX_OUTPUT_DIR, 'geometry_fix_summary.csv')

if os.path.exists(SUMMARY_CSV):
    summary_df = pd.read_csv(SUMMARY_CSV)
    print(f"Loaded cached results for {len(summary_df)} galaxies.")
else:
    rows = []
    t_start = time.time()
    n_missing = 0
    for _, r in df_agn.iterrows():
        row = process_galaxy(r['ID'], r['field'], r[z_col], r['fracAGN'], r['i'])
        if row is None:
            n_missing += 1
            continue
        rows.append(row)
        if len(rows) % 500 == 0:
            print(f"  {len(rows)}/{len(df_agn)} galaxies processed ({time.time() - t_start:.0f}s elapsed)")

    summary_df = pd.DataFrame(rows)
    summary_df.to_csv(SUMMARY_CSV, index=False)
    print(f"Done: {len(summary_df)} galaxies processed, {n_missing} missing FITS files skipped, "
          f"{time.time() - t_start:.0f}s total.")

## 2. Headline result: does matching CIGALE's own geometry fix the Quiescent-region asymmetry? (Figure 1)

Compares Type1, Type2, and Matched reconstructions against the real
`UV_Full`/`VJ_Full` classification, broken down by true UVJ region - the
same breakdown that revealed the original asymmetry.

In [ ]:
def bootstrap_ci(values, stat_fn=np.mean, n_boot=2000, seed=42, ci=(2.5, 97.5)):
    """Non-parametric percentile bootstrap CI (docs/figure9_10_bootstrap_methodology.md convention)."""
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.nan, np.nan, np.nan
    point = stat_fn(values)
    rng = np.random.default_rng(seed)
    boot = rng.choice(values, size=(n_boot, len(values)), replace=True)
    boot_stat = stat_fn(boot, axis=1)
    lo, hi = np.percentile(boot_stat, ci)
    return point, lo, hi


catalog_cols = summary_df.merge(df_cig[['ID', 'UV_Full', 'VJ_Full']], on='ID', how='left')
catalog_cols['cls_full'] = photometry.classify_uvj(catalog_cols['VJ_Full'], catalog_cols['UV_Full'])

METHODS = ['Type1', 'Type2', 'Matched']
region_names = {0: 'Quiescent', 1: 'Star-forming', 2: 'Dusty'}

agreement_table = {}
for mname in METHODS:
    ok = catalog_cols[f'cls_recombined_{mname}'] != -1
    agreement_table[mname] = {}
    for cid in [0, 1, 2]:
        mask = ok & (catalog_cols['cls_full'] == cid)
        correct = (catalog_cols.loc[mask, f'cls_recombined_{mname}'] == cid).astype(float)
        p, lo, hi = bootstrap_ci(correct, np.mean, seed=100 + cid)
        agreement_table[mname][cid] = (p, lo, hi, mask.sum())

fig, ax = plt.subplots(figsize=visualization.PASA_WIDE)
x = np.arange(3)
width = 0.25
for i_m, mname in enumerate(METHODS):
    pts = [agreement_table[mname][cid][0] for cid in [0, 1, 2]]
    los = [agreement_table[mname][cid][1] for cid in [0, 1, 2]]
    his = [agreement_table[mname][cid][2] for cid in [0, 1, 2]]
    err = [np.array(pts) - np.array(los), np.array(his) - np.array(pts)]
    ax.bar(x + (i_m - 1) * width, pts, width, yerr=err, capsize=2,
           color=METHOD_COLORS[mname], label=mname)
ax.set_xticks(x)
ax.set_xticklabels([region_names[c] for c in [0, 1, 2]])
ax.set_ylabel('UVJ classification agreement (reconstructed vs Full)')
ax.set_ylim(0, 1.05)
ax.set_title('Fixing the Quiescent-region asymmetry: Type1/Type2 vs CIGALE-matched geometry')
ax.legend(fontsize=8)

plt.tight_layout()
fig.savefig(os.path.join(FIX_OUTPUT_DIR, 'Figure1_geometry_fix_agreement.png'), dpi=300, bbox_inches='tight')
plt.show()

print("Per-region classification agreement (point [95% CI], n):")
for mname in METHODS:
    print(f"\n{mname}:")
    for cid in [0, 1, 2]:
        p, lo, hi, n = agreement_table[mname][cid]
        print(f"  {region_names[cid]:14s} n={n:5d}  {p:.1%} [{lo:.1%}, {hi:.1%}]")

for mname in METHODS:
    ok = catalog_cols[f'cls_recombined_{mname}'] != -1
    fp = ok & (catalog_cols['cls_full'] != 0) & (catalog_cols[f'cls_recombined_{mname}'] == 0)
    fn = ok & (catalog_cols['cls_full'] == 0) & (catalog_cols[f'cls_recombined_{mname}'] != 0)
    print(f"\n{mname}: false-positive Quiescent rate = {fp.sum() / ok.sum():.2%} ({fp.sum()}/{ok.sum()}), "
          f"false-negative Quiescent rate (of true Quiescent) = "
          f"{fn.sum() / (catalog_cols['cls_full'] == 0).sum():.1%} ({fn.sum()}/{(catalog_cols['cls_full'] == 0).sum()})")

## 3. Does the fix also improve flux-level and colour-level fidelity overall? (Figure 2)

The classification-agreement fix (Figure 1) is the headline result, but a
"fix" that only helps classification while making the underlying flux/colour
reconstruction worse would be suspicious - this checks both.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=visualization.PASA_WIDE)

ax = axes[0]
for mname in METHODS:
    vals = summary_df[f'max_abs_resid_{mname}'].replace([np.inf, -np.inf], np.nan).dropna()
    vals = vals[vals > 0]
    ax.hist(np.log10(vals), bins=50, color=METHOD_COLORS[mname], alpha=0.5, label=mname, density=True,
            histtype='step', linewidth=1.5)
ax.set_xlabel('log10(per-galaxy max |relative residual|), wavelength > 1300 A')
ax.set_ylabel('Density')
ax.set_title('Tier 1: flux reconstruction fidelity')
ax.legend(fontsize=7)

ax = axes[1]
for mname in METHODS:
    duv = catalog_cols[f'UV_recombined_{mname}'] - catalog_cols['UV_Full']
    dvj = catalog_cols[f'VJ_recombined_{mname}'] - catalog_cols['VJ_Full']
    combined = pd.concat([duv.abs(), dvj.abs()])
    ax.hist(combined.clip(upper=0.5), bins=60, histtype='step', color=METHOD_COLORS[mname],
            label=mname, density=True, linewidth=1.5)
ax.set_xlabel('|dUV| or |dVJ| (mag)')
ax.set_ylabel('Density')
ax.set_title('Tier 2: colour reconstruction fidelity')
ax.legend(fontsize=7)

plt.tight_layout()
fig.savefig(os.path.join(FIX_OUTPUT_DIR, 'Figure2_geometry_fix_flux_colour_fidelity.png'), dpi=300, bbox_inches='tight')
plt.show()

for mname in METHODS:
    duv = catalog_cols[f'UV_recombined_{mname}'] - catalog_cols['UV_Full']
    dvj = catalog_cols[f'VJ_recombined_{mname}'] - catalog_cols['VJ_Full']
    m_uv, lo_uv, hi_uv = bootstrap_ci(np.abs(duv), np.mean, seed=201)
    m_vj, lo_vj, hi_vj = bootstrap_ci(np.abs(dvj), np.mean, seed=211)
    med_resid = summary_df[f'resid_median_{mname}'].median()
    print(f"{mname}: mean|dUV|={m_uv:.4f} [{lo_uv:.4f},{hi_uv:.4f}], "
          f"mean|dVJ|={m_vj:.4f} [{lo_vj:.4f},{hi_vj:.4f}], "
          f"population median flux residual={med_resid:+.4f}")

## 4. Verification: single-galaxy spot-check

Picks a galaxy from the Quiescent region that Type1 or Type2 misclassified
but Matched gets right, and overlays all three reconstructions against the
real Full SED, for a direct visual explanation of *why* the matched
geometry works better (how much optical/UV flux each AGN template
contributes relative to the host).

In [ ]:
candidates = catalog_cols[
    (catalog_cols['cls_full'] == 0)
    & (catalog_cols['cls_recombined_Matched'] == 0)
    & (catalog_cols['cls_recombined_Type1'] != 0)
]
if len(candidates) == 0:
    candidates = catalog_cols[(catalog_cols['cls_full'] == 0) & (catalog_cols['cls_recombined_Matched'] == 0)]

if len(candidates) == 0:
    print("No suitable spot-check galaxy found.")
else:
    spot = candidates.iloc[0]
    spot_id, field = spot['ID'], spot['field']
    path = _fits_path(spot_id, field)
    full_sed = data_io.read_cigale_best_model(path, redshift=spot['redshift'], restframe=True)
    host_sed = analysis.decompose_cigale_sed(full_sed, target='host')
    matched_template = MATCHED_TEMPLATES[int(round(spot['agn_i']))]

    fig, ax = plt.subplots(figsize=visualization.PASA_WIDE)
    ax.loglog(full_sed['lambda (Angstroms)'], full_sed['L_lambda_total'], color='k', lw=1.5, label='Full (CIGALE)')
    ax.loglog(host_sed['lambda (Angstroms)'], host_sed['Total Flux (erg/s/cm^2/Angstrom)'], color='gray', ls='--',
               label='Host (decomposed)')
    for mname, tmpl in [('Type1', FIXED_TEMPLATES['Type1']), ('Type2', FIXED_TEMPLATES['Type2']),
                         ('Matched', matched_template)]:
        composite = composite_math.create_composite_sed(tmpl, host_sed, spot['alpha_theory'])
        ax.loglog(composite['lambda (Angstroms)'], composite['Total Flux (erg/s/cm^2/Angstrom)'],
                   color=METHOD_COLORS[mname], ls=':', label=f'Host + {mname}')
    ax.axvspan(3000, 13000, color='yellow', alpha=0.08, label='approx. U-J restframe range')
    ax.set_xlabel('Restframe wavelength (A)')
    ax.set_ylabel('Flux (L_lambda_total units)')
    ax.set_title(f"{spot_id}: true Quiescent (fracAGN={spot['fracAGN']:.2f}, "
                 f"CIGALE agn.i={spot['agn_i']:.0f})")
    ax.legend(fontsize=7)
    plt.tight_layout()
    fig.savefig(os.path.join(FIX_OUTPUT_DIR, 'Figure3_spotcheck_overlay.png'), dpi=300, bbox_inches='tight')
    plt.show()

    print(f"cls_full=Quiescent; cls_recombined: Type1={spot['cls_recombined_Type1']}, "
          f"Type2={spot['cls_recombined_Type2']}, Matched={spot['cls_recombined_Matched']} (0=Quiescent)")

## 5. Conclusion

Matching each galaxy's reconstruction to CIGALE's own best-fit SKIRTOR
geometry (`p=1.0, q=1.0`, and `i=30` or `i=70` chosen per galaxy) instead of
the paper's fixed Type 1 (`p=0.5, q=0, i=0`) / Type 2 (`p=0.5, q=0, i=90`)
templates directly resolves the Quiescent-region asymmetry found in
`CIGALE_Decomposition_Validation.ipynb`, without requiring per-galaxy
fitting of `alpha` or any other free parameter - `alpha_theory =
fracAGN / (1 - fracAGN)` is unchanged. Because CIGALE's fitting run only
ever explored two discrete geometries, this fix is exactly as cheap as the
original Type1/Type2 approach (two template reads total, reused across the
whole population).

**Caveat:** this specific fix (two fixed geometries, selected by `agn.i`) is
a consequence of *this* CIGALE run's configuration having explored only two
AGN geometries. A future CIGALE run with a broader AGN parameter grid (more
`t`, `p`, `q`, `oa`, `R` values, not just `i`) would need the more general
per-galaxy exact-match approach (reading all six parameters from each
galaxy's own header) rather than this two-template shortcut - the
`process_galaxy` structure here generalizes directly to that case by
building a larger `MATCHED_TEMPLATES` cache keyed on the full parameter
tuple instead of `i` alone.

This notebook does not modify `CIGALE_Decomposition_Analysis.ipynb`,
`CIGALE_Decomposition_Validation.ipynb`, or any code in the `glass` package
- adopting this fix in the paper's actual pipeline (e.g. in
`composite_math.create_composite_sed` calls elsewhere, or in
`config.SKIRTOR_TYPE1_PARAMS`/`TYPE2_PARAMS`) is a decision for the paper's
author, not made here.